# Predictions from Baryon Power Spectrum

Because the probability of conversion is a function of $m_\gamma^2$, which is in turn a function of the baryon power spectrum, we can predict the power spectrum of the probability of conversion (as well as the global signal) from $P_{bb}$, which we can simulate from `CLASS`. Of course, this assumes that we can neglect perturbations in the free electron fraction, which is only true about $z\approx 20$. 

Description: This module computes analytic angular power spectra (Cls) and two point functions for dark photon perturbations in the low mass regime, using the lognormal PDF for density fluctuations. It saves the results for different dark photon mass values (mA) to specified output files. The user needs to install this https://github.com/smsharma/dark-photons-perturbations first. The script in this directory does the same thing as this notebook, but is more convenient for running many times.

To do this, we first import a power spectrum $P_{\rm bb}$ from `CLASS' and compute
$$ \begin{align} \sigma_{\rm b}^2(z) &= \int \frac{d^3\vec{k}}{(2\pi)^3}P_{\rm bb}(k,z).
\end{align}$$


In [ ]:
import os, sys
import pickle
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec
from matplotlib import ticker
from scipy.optimize import fsolve
import seaborn as sns
import numpy as np
from scipy.integrate import cumulative_trapezoid, quad
from tqdm import *
from scipy.interpolate import interp1d
import scipy.special as sp
import hankel

grf_path = "/home/bakerem/dark-photons-perturbations"
sys.path.append(grf_path)

from grf.grf import PerturbedProbability, FIRAS
from grf.pk_interp import PowerSpectrumGridInterpolator
from grf.units import *

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2
# Load plot settings

from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']
# Where to save plots
plots_dir = "../plots/"
z_compute_ary = np.geomspace(-3, 3, 500)
k_ary = np.logspace(-4, np.log10(1e3), 500)

log_pk_lin_baryon_grid_ary = np.zeros((len(z_compute_ary), len(k_ary)))

for i_z, z_compute in enumerate(z_compute_ary):
    ary = np.load(grf_path+"/data/pk_arys/p_k_k_max_5000_z_" + str(i_z) + ".npz")
    interp_lin_baryon = interp1d(np.log10(ary['k_ary']), np.log10(ary['Pk_b_ary']), bounds_error=False, fill_value="extrapolate")
log_pspec = PowerSpectrumGridInterpolator("lin_baryon")

In [ ]:
# this computes the total probability of conversion only considering crossings above z=20. 
# This is a *severe* underestimate of the total probability of conversion, but it is useful for comparison with
# the 21cmFAST results. We can check the predicted probability of conversion from 
# 1) the simulation directly
# 2) the m_gamma^2 pdf from the simulation
# 3) the baryon power spectrum from CLASS (this code).
# If we get the same results in each case we can be confident that the simulation is working correctly. 
# However, note that we stay away from recombination because of the substantial uncertainties and difficulties
# in modeling the power spectrum at these redshifts.

prob = PerturbedProbability(log_pspec)

# Non-linear matter power spectrum. 
pspec_lin_baryon = PowerSpectrumGridInterpolator("lin_baryon")

# Class containing results with linear baryon spectrum. 
prob = FIRAS(pspec_lin_baryon)

one_plus_delta_bound = 1e2  # Fiducial bound
low_mA_list = np.geomspace(1.5e-14, 1e-13, 15)
halo_mA_list = np.geomspace(1e-13, 1e-11, 25)
mA_list = np.concatenate([halo_mA_list, low_mA_list]) * eV
mA_list = np.geomspace(1e-16, 1e-9, 300) * eV

In [ ]:
P_tot_an_ary = np.zeros_like(mA_list)
rs_array = np.geomspace(0.001, 1000, 1000)
rs_array = rs_array[(rs_array < 5) | (rs_array > 35)] # 21cmFAST computation 

# Log-normal PDF with \delta bound

P_tot_bounded_ary = np.zeros_like(mA_list)

for i_m_Ap, m_Ap in enumerate(tqdm(mA_list)):
    T0 = 2.73 * 8.62e-5
    xobs = 0.0251/(1+17) # at z=17
    omega_ary = xobs * T0 * eV
    dPdrs_array = prob._dP_dz(z_ary=rs_array, m_Ap=m_Ap, k_min=1e-3, k_max=1e3, omega=omega_ary, pdf='lognormal', one_plus_delta_bound=one_plus_delta_bound)[0][0]
    dPdrs_array_cum = np.concatenate([dPdrs_array[rs_array < 6], dPdrs_array[rs_array > 20]])
    # P_tot_bounded_ary[i_m_Ap] = np.trapz(dPdrs_array_cum, rs_array_cum)
    P_tot_bounded_ary[i_m_Ap] = np.trapz(dPdrs_array, rs_array)


In [ ]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

bt_limit = -100
z = 17
Tgamma0_mK = 2.73 * 1000 #mK

z_index = np.where(rs_array == find_nearest(rs_array, z))[0]
bt_at_z = 0 #bt[z_index]
xobs = 0.0251/(1+z)
xe = prob.x_e(rs_array)


lognorm_lims = []
for i_m_Ap, m_Ap in enumerate(mA_list):
    def find_lognorm_eps(eps):
        P_lognorm = P_tot_bounded_ary[i_m_Ap] * eps**2 
        return bt_limit - (bt_at_z - Tgamma0_mK * P_lognorm)
    
    lognorm_sol = fsolve(find_lognorm_eps, 1e-6,maxfev=10000, xtol=1e-10, full_output=True)
    z_star = lognorm_sol[0]
    # checks to see if there actually was a solution found
    if lognorm_sol[2] != 1 or z_star > rs_array.max():
        lognorm_eps = 100
    else:
        lognorm_eps = lognorm_sol[0][0]
    lognorm_lims.append(np.array([m_Ap/eV, lognorm_eps]), )

lognorm_lims = np.array(lognorm_lims)
# full_anal_lims = np.array(full_anal_lims)


In [ ]:
homo_lims = []
for i_m_Ap, m_Ap in enumerate(mA_list):    
    mA = m_Ap / eV
    mgamma2 = prob.m_A_sq(rs_array, omega_ary) / eV**2
    T0 = 2.73 * 8.62e-5 # to eV
    H0 = 67.66 * 6.57895e-16 * 3.241e-20

    # compute homogeneous prediction from code
    mgamma2_interp =interp1d(rs_array, mgamma2,bounds_error=False, fill_value="extrapolate")
    if mA < 1.5e-13 and mA>np.sqrt(mgamma2[0]):
        x0 = 1
    else:
        x0 = 100
    output = fsolve(lambda z: np.sqrt(mgamma2_interp(z)) - mA, x0, maxfev=10000, full_output=True)
    z_star = output[0]
    # checks to see if there actually was a solution found
    if output[2] != 1 or z_star > rs_array.max():
        Ptot_prefac = 1e-100
    else:
        Ptot_prefac = np.pi * mA**2 / (3 *  xobs * T0 * (1+z_star[0]) * prob.cosmo.H(z_star).value[0] * Kmps / Mpc / eV)   
     
    homo_eps = np.sqrt(-(bt_limit - bt_at_z) /(Tgamma0_mK * Ptot_prefac))
    homo_lims.append([mA, homo_eps])

homo_lims = np.array(homo_lims)

In [ ]:
m_Ap_DP, lim_DP = np.transpose(np.loadtxt("/home/bakerem/dark_photon_21cm_constraints/halo_data/data_from_papers/fiducial_DP_FIRAS_one_plus_delta_1e2.csv", skiprows=2, delimiter=','))
mA_index = np.where(homo_lims[:,0] == find_nearest(homo_lims[:,0],np.sqrt(prob.m_A_sq(370, 1)/eV**2)))[0][0]
plot_lognorm_lims = lognorm_lims.copy()
plot_lognorm_lims[mA_index:,1] = homo_lims[mA_index:, 1]

plt.plot(plot_lognorm_lims[:,0], plot_lognorm_lims[:,1], label="Lognormal")
# plt.plot(homo_lims[:,0], homo_lims[:,1], label="Homogeneous")

plt.fill_between(m_Ap_DP, lim_DP, 1e-2, alpha=0.2, color=cols_default[3])
plt.xscale("log")
plt.yscale("log")
plt.ylim(top=1e-2)
plt.xlabel(r"$m_{A'}$ [eV]")
plt.ylabel(r"$\varepsilon$")
plt.xticks()
plt.xlim(1e-16, 1e-9)
plt.text(5e-13,1e-4, r"FIRAS Limit", fontsize=16)
# plt.title(r"Limits from Conversions before $z=1000$")
plt.legend(loc="lower left")
ax = plt.gca()

locmaj = ticker.LogLocator(base=10,numticks=12) 
ax.xaxis.set_major_locator(locmaj)

locmin = ticker.LogLocator(base=10.0,subs=(0.1,0.2, 0.3, 0.4,0.5,0.6,0.7,0.8,0.9),numticks=12)
ax.xaxis.set_minor_locator(locmin)
ax.xaxis.set_minor_formatter(ticker.NullFormatter())

locmaj = ticker.LogLocator(base=10,numticks=12) 
ax.yaxis.set_major_locator(locmaj)

locmin = ticker.LogLocator(base=10.0,subs=(0.1,0.2, 0.3, 0.4,0.5,0.6,0.7,0.8,0.9), numticks=12)
ax.yaxis.set_minor_locator(locmin)
ax.yaxis.set_minor_formatter(ticker.NullFormatter())


In [ ]:
plt.plot(zs, np.sqrt(mgamma2))
plt.axhline(4e-12)
plt.yscale("log")
# plt.xlim(0.01, 0.5)
# plt.ylim(1e-14, 1e-13)
